In [ ]:
import numpy as np

In [ ]:
class Element:
  def setArea(self):
    self.area = 0.5*(
        (self.x2-self.x1)*(self.y3-self.y1)
        - (self.x3-self.x1)*(self.y2-self.y1)
    )

  def __init__(self, nodes):
    self.x1 = nodes[0][0]
    self.x2 = nodes[1][0]
    self.x3 = nodes[2][0]

    self.y1 = nodes[0][1]
    self.y2 = nodes[1][1]
    self.y3 = nodes[2][1]

    self.setArea()

    self.P = []
    self.Q = []

  def calculateP(self):
    P1 = self.y2 - self.y3
    P2 = self.y3 - self.y1
    P3 = self.y1 - self.y2

    self.P = [P1, P2, P3]

  def calculateQ(self):
    Q1 = -(self.x2 - self.x3)
    Q2 = -(self.x3 - self.x1)
    Q3 = -(self.x1 - self.x2)

    self.Q = [Q1, Q2, Q3]

  def calculateCoefficientsMatrix(self):
    self.calculateP()
    self.calculateQ()

    coefficientsMatrix = np.empty((3,3))

    for i in range(3):
      for j in range(3):
        coefficientsMatrix[i][j] = (
            self.P[i]*self.P[j]
            + self.Q[i]*self.Q[j]
        )/self.area

    return coefficientsMatrix

In [ ]:
class FEMSolver:
  def __init__(self, nodes, elementsNodeIndices):
    self.nodes = nodes
    self.elementsNodeIndices = elementsNodeIndices

  def calculateElementCoefficientMatrices(self):
    elementCoefficientMatrices = []
    for elementNodeIndices in self.elementsNodeIndices:
      element = Element(
          [self.nodes[elementNodeIndices[0]-1],
           self.nodes[elementNodeIndices[1]-1],
           self.nodes[elementNodeIndices[2]-1]
           ])
      elementCoefficientMatrices.append(element.calculateCoefficientsMatrix())

    return elementCoefficientMatrices

#Sheet 2 (4)

##Numerical solution

In [ ]:
nodes = [
     (2,1),
     (4,1),
     (4,2),
     (2,2),
     (0,2),
     (0,1),
     (0,0),
     (2,0),
     (4,0),
     (6,1),
     (6,2),
     (6,3),
     (4,3),
     (2,3),
     (0,3),
     (6,0)
]
elementNodes = [
    (1,2,3),
    (1,3,4),
    (1,4,6),
    (1,6,7),
    (1,7,8),
    (2,1,8),
    (2,8,9),
    (2,9,10),
    (2,10,11),
    (3,2,11),
    (3,11,12),
    (3,12,13),
    (4,3,13),
    (4,13,14),
    (4,14,5),
    (4,5,6),
    (5,14,15),
    (9,10,16)
]

In [ ]:
LERectangleFEMSolver = FEMSolver(nodes, elementNodes)

In [ ]:
elementCoefficientMatrices = LERectangleFEMSolver.calculateElementCoefficientMatrices()

In [ ]:
elementCoefficientMatrices[18-1]

array([[-1., -0.,  1.],
       [-0., -4.,  4.],
       [ 1.,  4., -5.]])

In [ ]:
Cff = np.array(
    [
      [20,-2,0,-8],
      [-2,20,-8,0],
      [0,-8,20,-2],
      [-8,0,-2,20]
  ]
)
print(Cff.shape)

Cfp = np.array(
    [
     [0,-2,0,-8,0,0,0,0,0,0,0,0],
     [0,0,0,0,-8,-2,0,0,0,0,0,0],
     [0,0,0,0,0,0,-2,0,-8,0,0,0],
     [-2,0,0,0,0,0,0,0,0,-8,0,0]
    ]
)
print(Cfp.shape)

Vp = np.array([-2,-2,-1,0,0,5,5,7.5,10,10,4,2.5])
print(Vp.shape)

Vf = -np.dot(np.dot(np.linalg.inv(Cff),Cfp),Vp)

Vf

(4, 4)
(4, 12)
(12,)


array([2.26153846, 3.26153846, 6.33846154, 5.33846154])

##Analytical solution

In [ ]:
v1 = 0
v2 = 0
v3 = 0
v4 = 0

I = 50
for N in range(I):
  n = 2*N+1
  v1 += (4/(np.pi*n))*(
      5*np.sin(2*np.pi*n/3)*np.sinh(np.pi*n/3)/np.sinh(2*np.pi*n)
      + 10*np.sin(np.pi*n/3)*np.sinh(np.pi*n/6)/np.sinh(np.pi*n/2)
      -2*np.sin(4*np.pi*n/3)*np.sinh(np.pi*n/3)/np.sinh(2*np.pi*n)
  )

  v2 += (4/(np.pi*n))*(
      5*np.sin(4*np.pi*n/3)*np.sinh(np.pi*n/3)/np.sinh(2*np.pi*n)
      + 10*np.sin(2*np.pi*n/3)*np.sinh(np.pi*n/6)/np.sinh(np.pi*n/2)
      -2*np.sin(2*np.pi*n/3)*np.sinh(np.pi*n/3)/np.sinh(2*np.pi*n)
  )

  v3 += (4/(np.pi*n))*(
      5*np.sin(4*np.pi*n/3)*np.sinh(2*np.pi*n/3)/np.sinh(2*np.pi*n)
      + 10*np.sin(2*np.pi*n/3)*np.sinh(np.pi*n/3)/np.sinh(np.pi*n/2)
      -2*np.sin(2*np.pi*n/3)*np.sinh(2*np.pi*n/3)/np.sinh(2*np.pi*n)
  )

  v4 += (4/(np.pi*n))*(
      5*np.sin(2*np.pi*n/3)*np.sinh(2*np.pi*n/3)/np.sinh(2*np.pi*n)
      + 10*np.sin(np.pi*n/3)*np.sinh(np.pi*n/3)/np.sinh(np.pi*n/2)
      -2*np.sin(4*np.pi*n/3)*np.sinh(2*np.pi*n/3)/np.sinh(2*np.pi*n)
  )
print(v1, v2, v3, v4)

2.6503835162192395 2.578349498794104 5.748184255566955 5.978734435919397


#Sheet 2 (5)

##Numerical solution

In [ ]:
nodes = [
     (1,1),
     (2,1),
     (2,2),
     (1,2),
     (0,1),
     (0,0),
     (1,0),
     (2,0),
     (3,1),
     (3,2)
]
elementNodes = [
    (1,2,3),
    (1,3,4),
    (1,4,5),
    (1,5,6),
    (1,6,7),
    (2,1,7),
    (2,7,8),
    (2,8,9),
    (2,9,10),
    (2,10,3)
]

In [ ]:
LERectangleFEMSolver = FEMSolver(nodes, elementNodes)

In [ ]:
elementCoefficientMatrices = LERectangleFEMSolver.calculateElementCoefficientMatrices()

In [ ]:
elementCoefficientMatrices[10-1]

array([[ 2.,  0., -2.],
       [ 0.,  2., -2.],
       [-2., -2.,  4.]])

In [ ]:
Tff = np.array([[3/2,1/6], [1/6,3/2]])
Cff = np.array([[10,-4], [-4,16]])

np.dot(np.linalg.inv(Tff),Cff)

array([[ 7.05, -3.9 ],
       [-3.45, 11.1 ]])